In [ ]:
# Import necessary libraries
import os
import sys
import shutil
import glob

import numpy as np
import yaml
from tqdm.contrib.concurrent import thread_map# or thread_map

In [ ]:
def create_project_file(file_pth: str, dataset_pth: str, label_mapping: dict, val_folder: str = "images/val") -> None:
    data = {
        "path": dataset_pth,
        "train": "images/train",
        "val": val_folder,
        "kpt_shape": [1, 2],
    }

    with open(file_pth, "wt") as f:
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
    
    with open(file_pth, "at") as f:
        yaml.dump({"names": label_mapping}, f, sort_keys=False)

In [ ]:
def get_files_glob(directory, ext="*.txt"):
    pattern = os.path.join(directory, '**', ext)
    txt_files = glob.glob(pattern, recursive=True)
    return txt_files

In [ ]:
MARGIN = 0.10

BASE_PTH = "/home/klymy/data/cam4_revB/yolo_dataset"
BASE_DATASET = "batch_testAccuracy"

FILTERED_DATASET = f"{BASE_DATASET}_margin_{int(MARGIN * 100)}"

SUBSETS = ["train", "val", "holdout"]

In [ ]:
src_pth = os.path.join(BASE_PTH, BASE_DATASET)
dst_pth = os.path.join(BASE_PTH, FILTERED_DATASET)
if os.path.exists(dst_pth):
    shutil.rmtree(dst_pth)
shutil.copytree(src_pth, dst_pth)

cache_files = get_files_glob(dst_pth, "*.cache")
for i in cache_files:
    os.remove(i)

In [ ]:
# create project file
create_project_file(os.path.join(dst_pth, "project.yaml"), dst_pth, {0: "person"})
create_project_file(os.path.join(dst_pth, "project_holdout.yaml"), dst_pth, {0: "person"}, val_folder="images/holdout")

In [ ]:
labels_pths = get_files_glob(os.path.join(dst_pth, "labels"))
len(labels_pths)

In [ ]:
def get_min_margin(x, y, widths, heights):
    # Calculate the margin sizes
    left_margin = x + widths / 2  # Distance from right side of the box to the left edge of the frame
    right_margin = 1 - (x - widths / 2)  # Distance from left side of the box to the right edge of the frame
    top_margin = y + heights / 2  # Distance from bottom side of the box to the top edge of the frame
    bottom_margin = 1 - (y - heights / 2)  # Distance from top side of the box to the bottom edge of the frame
    
    return np.min([left_margin, right_margin, top_margin, bottom_margin])

def process_file(pth):
    filtered_lines = []

    count = 0
    count_filtered = 0
    with open(pth, "rt") as f:
        for line in f.readlines():
            count +=1
            _, x_c, y_c, w, h, _, _ = [float(i) for i in line.split()]
            
            min_margin = get_min_margin(x_c, y_c, w, h)
            if min_margin > MARGIN:
                filtered_lines.append(line)
                count_filtered += 1

    with open(pth, "wt") as f:
        f.writelines(filtered_lines)

    return count, count_filtered

In [ ]:
r = thread_map(process_file, labels_pths, max_workers=8)

count = 0
count_filtered = 0
for i, j in r:
    count += i
    count_filtered += j

print(f"Margin {MARGIN} removes {(count - count_filtered) / count * 100:.02f}% bboxes")